# 02 - Preprocessing
Goal: clean the ASEADOS-SDN-IoT data and split it into train and test sets.

In [30]:
import numpy as np
import pandas as pd

CSV_PATH = '../data/ASEADOS_SDN_IoT.csv'
df = pd.read_csv(CSV_PATH, low_memory=False)
df.columns = df.columns.str.strip()

print('Start:',df.shape)

Start: (457044, 84)


In [31]:
mask = df['Flow Duration'] < 0 
print(mask.head())

0    False
1    False
2    False
3    False
4    False
Name: Flow Duration, dtype: bool


In [32]:
print('Negative Flow Duration rows:',mask.sum())
print('Percent:',mask.sum() / len(df) * 100)

Negative Flow Duration rows: 2
Percent: 0.0004375946298387026


In [33]:
print(pd.crosstab(df['Protocol'], df['Label']))

Label     Benign   Bot   DDoS     Dos  Prob
Protocol                                   
0          42740  9090  51465   16075  7920
6         216512     0      0  111654     0
17          1545     0      0      43     0


Check B: Bot, DDoS, and probe flows all have Protocol = 0. Since "Protocol" is a shortcut, it will be removed.

In [34]:
id_cols = ["Flow ID", "Src IP", "Src Port","Dst IP", "Dst Port", "Timestamp"]
df.drop(columns=id_cols).duplicated().sum()

np.int64(29993)

In [35]:
mask = df.drop(columns=id_cols).duplicated()
dup_rows = df[mask]
print(dup_rows['Label'].value_counts())

Label
Benign    27046
Dos        1265
DDoS       1195
Prob        354
Bot         133
Name: count, dtype: int64


Most of the repetitions occur during normal traffic. This makes sense, as IoT devices constantly send the same small messages. Removing them causes a slight decrease in the "benign" category, but that’s not an issue.

In [36]:
drop_cols = id_cols + ["Protocol"]
df = df.drop(columns=drop_cols)
print('After dropping ID columns + Protocol:', df.shape)

After dropping ID columns + Protocol: (457044, 77)


In [37]:
nunique = df.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
df = df.drop(columns=constant_cols)
print('After dropping constant columns:',df.shape)

After dropping constant columns: (457044, 63)


First, we convert the infinite values ​​into null values ​​so that we can delete them all at once.

In [38]:
df = df.replace([np.inf, -np.inf],np.nan)
df = df.dropna()
print('After removing NaN/inf rows:',df.shape)

After removing NaN/inf rows: (456974, 63)


In [39]:
df = df[df['Flow Duration'] >= 0]
print('After removing negative durations:',df.shape)

After removing negative durations: (456972, 63)


In [40]:
df = df.drop_duplicates()
print('After removing duplicates:',df.shape)
print(df['Label'].value_counts())

After removing duplicates: (427042, 63)
Label
Benign    233745
Dos       126506
DDoS       50269
Bot         8957
Prob        7565
Name: count, dtype: int64


## Cleaning summary

| Step | Rows | Columns |
|---|---|---|
| Start | 457,044 | 84 |
| Drop 6 ID columns + Protocol | 457,044 | 77 |
| Drop 14 constant columns | 457,044 | 63 |
| Remove NaN/inf rows | 456,974 | 63 |
| Remove negative durations | 456,972 | 63 |
| Remove duplicates | 427,042 | 63 |

In [46]:
X = df.drop(columns='Label')
y_multi = df['Label']
y_binary = (df['Label'] != 'Benign').astype(int)
print('X:', X.shape)
print(y_binary.value_counts())

X: (427042, 62)
Label
0    233745
1    193297
Name: count, dtype: int64
